**Install Dependencies**

In [ ]:
!pip install biopython pandas requests tqdm

**Main Script**

Upload your Input file labelled as Maxquant.tsv into the session storage and run the next block.

In [ ]:
import pandas as pd
import re
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm import tqdm

# -----------------------------
# CONFIG
# -----------------------------
INPUT_FILE = "Maxquant.txt"
OUTPUT_TSV = "output.tsv"

# -----------------------------
# HELPERS
# -----------------------------

def clean_sequence(seq):
    """Keep only standard amino acids (removes modifications)."""
    return re.sub(r'[^ACDEFGHIKLMNPQRSTVWY]', '', str(seq).upper())


def extract_first_uniprot(protein_field):
    """Extract first UniProt ID from Proteins column."""
    if pd.isna(protein_field):
        return None
    return str(protein_field).split(";")[0]


def compute_properties(seq):
    """Compute peptide properties using Biopython."""
    if len(seq) == 0:
        return None

    analysis = ProteinAnalysis(seq)
    aa_freq = analysis.amino_acids_percent  # correct for new Biopython

    return {
        "length": len(seq),
        "mw": analysis.molecular_weight(),
        "gravy": analysis.gravy(),
        "pI": analysis.isoelectric_point(),
        **{f"freq_{aa}": aa_freq.get(aa, 0) for aa in "ACDEFGHIKLMNPQRSTVWY"}
    }

# -----------------------------
# LOAD MAXQUANT FILE
# -----------------------------
mq = pd.read_csv(INPUT_FILE, sep="\t")

# Keep only relevant columns
mq = mq[["Sequence", "Proteins"]].copy()

# -----------------------------
# PROCESS
# -----------------------------
rows = []

for _, row in tqdm(mq.iterrows(), total=len(mq)):
    peptide_raw = row["Sequence"]
    proteins_field = row["Proteins"]

    peptide = clean_sequence(peptide_raw)
    uniprot_id = extract_first_uniprot(proteins_field)

    props = compute_properties(peptide)

    out = {
        "Proteins": uniprot_id,
        "Sequence": peptide
    }

    if props:
        for k, v in props.items():
            out[k] = v

    rows.append(out)

# -----------------------------
# SAVE OUTPUT
# -----------------------------
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_TSV, sep="\t", index=False)

print("Done! Saved to:", OUTPUT_TSV)